In [ ]:
import time
import gc
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig
from hep_tracking.models import ScipyCKDTree, HnswGraph, FaissIVFFlat
from hep_tracking.pipeline import PipelineEvaluator
from hep_tracking.features import compute_pair_features, create_pair_dataset, split_by_event
from hep_tracking.classifiers import XGBoostWrapper

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

In [ ]:
# --- Zbiory danych do ewaluacji ---
DATASETS_TO_TEST = [
    "hard_10k", 
    "hard_100k", 
    "hard_1M"
]

# --- Konfiguracja wyszukiwarek kandydatów---
RETRIEVERS_CONFIG = [
    KNNModelConfig("cKDTree (Exact)", ScipyCKDTree, {"workers": -1}),
    KNNModelConfig("HNSW (ANN_CPU)", HnswGraph, {"m": 16, "ef_construction": 200, "ef": 50}),
    KNNModelConfig("IVFFlat (ANN_GPU)", FaissIVFFlat, {"nprobe": 20, "use_gpu": True})
]

# --- Konfiguracja Trenowania Klasyfikatora ML (XGBoost) ---
TRAIN_CLASSIFIER = True
TRAIN_DATASET_NAME = "hard_1M"
CLASSIFIER_PARAMS = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 6,
    "n_jobs": -1
}
# --- Parametry potoku---
K_NEIGHBORS = 5
ML_PROB_THRESHOLD = 0.5

# Cięcia geometryczne dla potoku bez ML
GEOM_CUTS = {
    "max_delta_r": 20.0,    # Maksymalna różnica promienia
    "min_dot_product": 0.95 # Minimalny iloczyn skalarny
}

# --- Flagi uruchomieniowe  ---
RUN_GEOMETRIC_PIPELINE = True
RUN_ML_PIPELINE = False
RUN_ALL_PAIRS_BASELINE = False

In [ ]:
if TRAIN_CLASSIFIER:
    print(f"=== Rozpoczęcie przygotowania modelu na zbiorze: {TRAIN_DATASET_NAME} ===")
    train_path = DATA_DIR / f"dataset_{TRAIN_DATASET_NAME}.npz"
    train_dataset = TrackDataset.from_npz(train_path)

    print("1. Wyszukiwanie sąsiadów do budowy par uczących (cKDTree)...")
    ref_knn = ScipyCKDTree(workers=-1)
    ref_knn.fit(train_dataset.X)
    _, train_candidates = ref_knn.kneighbors(train_dataset.X, k=K_NEIGHBORS)

    print("2. Ekstrakcja cech fizycznych dla par...")
    t0_feat = time.perf_counter()
    dataset_pairs = create_pair_dataset(train_dataset, train_candidates, max_neg_ratio=5)
    time_feat = time.perf_counter() - t0_feat
    print(f"   -> Zbudowano zbiór {len(dataset_pairs.y)} par w {time_feat:.2f} s")

    train_pairs, val_pairs, _ = split_by_event(
        dataset=dataset_pairs, 
        train_size=0.8, 
        val_size=0.2, 
        seed=42
    )
    print(f"   -> Rozkład par - Train: {len(train_pairs.y)}, Val: {len(val_pairs.y)}")

    print("3. Trening klasyfikatora XGBoost...")
    classifier = XGBoostWrapper(**CLASSIFIER_PARAMS)
    
    t0_train = time.perf_counter()
    classifier.fit(train_pairs.X, train_pairs.y, X_val=val_pairs.X, y_val=val_pairs.y)
    time_train = time.perf_counter() - t0_train
    
    print(f"=== ZAKOŃCZONO! Całkowity czas jednorazowego treningu: {time_train:.2f} s ===")
    
else:
    classifier = None
    print("Pomijam przygotowanie ML (Flaga TRAIN_CLASSIFIER = False).")

In [ ]:
from IPython.display import display

print("\n=== ROZPOCZĘCIE TESTÓW POTOKÓW END-TO-END ===")
all_results = []

for dataset_name in DATASETS_TO_TEST:
    print(f"\nPrzetwarzanie zbioru: {dataset_name}")
    ds_path = DATA_DIR / f"dataset_{dataset_name}.npz"
    if not ds_path.exists():
        print(f"  [UWAGA] Plik {ds_path.name} nie istnieje. Pomijam.")
        continue
        
    full_dataset = TrackDataset.from_npz(ds_path)
    unique_events = np.unique(full_dataset.event_ids)
    print(f"  Znaleziono {len(unique_events)} zdarzeń (events) w zbiorze.")
    
    for event_id in unique_events:
        mask = full_dataset.event_ids == event_id
        event_ds = TrackDataset(
            X=full_dataset.X[mask],
            y=full_dataset.y[mask],
            event_ids=full_dataset.event_ids[mask]
        )
        
        n_hits = len(event_ds.X)
        nlist = min(100, int(np.sqrt(n_hits)))
        
        evaluator = PipelineEvaluator(
            dataset=event_ds,
            k_neighbors=K_NEIGHBORS,
            geom_cuts=GEOM_CUTS,
            ml_threshold=ML_PROB_THRESHOLD
        )
        
        for retriever_cfg in RETRIEVERS_CONFIG:
            if "IVF" in retriever_cfg.name:
                retriever_cfg.model_kwargs["nlist"] = nlist
                
            retriever = retriever_cfg.model_factory(**retriever_cfg.model_kwargs)
            
            if RUN_GEOMETRIC_PIPELINE:
                res = evaluator.run_geometric_pipeline(retriever)
                res.update({
                    "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits,
                    "Retriever": retriever_cfg.name, "Filter": "Geometric Cuts",
                    "Pipeline": f"{retriever_cfg.name} + GeomCuts"
                })
                all_results.append(res)
                
            if RUN_ML_PIPELINE and classifier is not None:
                res = evaluator.run_ml_pipeline(retriever, classifier)
                res.update({
                    "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits,
                    "Retriever": retriever_cfg.name, "Filter": "XGBoost",
                    "Pipeline": f"{retriever_cfg.name} + XGBoost"
                })
                all_results.append(res)
        
        if RUN_ALL_PAIRS_BASELINE and classifier is not None:
            if n_hits > 15000:
                print(f"  [POMINIĘTO] Zdarzenie {event_id} dla All-Pairs (zbyt wiele hitów: {n_hits}).")
            else:
                res = evaluator.run_all_pairs_pipeline(classifier)
                res.update({
                    "Dataset": dataset_name, "Event_ID": event_id, "Hits": n_hits,
                    "Retriever": "None (All Pairs)", "Filter": "XGBoost",
                    "Pipeline": "All-Pairs + XGBoost"
                })
                all_results.append(res)

df_results = pd.DataFrame(all_results)
if not df_results.empty:
    df_results["Physics_Quality"] = df_results["Purity"] * df_results["Efficiency"]

print("\n=== ZAKOŃCZONO WSZYSTKIE TESTY ===")
display(df_results.head())

In [ ]:
summary_df = df_results.groupby(["Dataset", "Pipeline"]).agg(
    Purity_Mean=("Purity", "mean"),
    Efficiency_Mean=("Efficiency", "mean"),
    Physics_Quality_Mean=("Physics_Quality", "mean"),
    Time_Retrieval_Mean=("Time_Retrieval_s", "mean"),
    Time_Features_Mean=("Time_Features_s", "mean"),
    Time_Filter_Mean=("Time_Filter_s", "mean"),
    Time_Total_Mean=("Time_Total_s", "mean")
).reset_index()

summary_df = summary_df.sort_values(by="Physics_Quality_Mean", ascending=False)

print("\n=== TABELA PODSUMOWUJĄCA: ŚREDNIE CZASY I METRYKI NA ZDARZENIE ===")
display(summary_df.style.format({
    "Purity_Mean": "{:.3f}", 
    "Efficiency_Mean": "{:.3f}", 
    "Physics_Quality_Mean": "{:.3f}",
    "Time_Retrieval_Mean": "{:.4f} s",
    "Time_Features_Mean": "{:.4f} s",
    "Time_Filter_Mean": "{:.4f} s",
    "Time_Total_Mean": "{:.4f} s"
}).background_gradient(subset=["Physics_Quality_Mean"], cmap="RdYlGn"))

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df_results, 
    x="Time_Total_s", 
    y="Physics_Quality", 
    hue="Pipeline", 
    alpha=0.3, 
    edgecolor=None,
    legend=False,
    s=50
)

sns.scatterplot(
    data=summary_df, 
    x="Time_Total_Mean", 
    y="Physics_Quality_Mean", 
    hue="Pipeline", 
    s=300, 
    edgecolor="black", 
    linewidth=2,
    marker="X"
)

plt.title("Physics Quality vs Total Time per Event", fontsize=16, fontweight='bold')
plt.xlabel("Total Time per Event (seconds)", fontsize=14)
plt.ylabel("Physics Quality (Purity x Efficiency)", fontsize=14)

plt.xscale("log")
plt.ylim(-0.05, 1.05)
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.legend(title="Pipeline (X = Mean)", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

output_file = PROJECT_ROOT / "plot.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
print(f"\nWygenerowano i zapisano wykres: {output_file.name}")

plt.show()